# assumpcheck example notebook

This notebook shows example usage of:

- `check_anova(...)`
- `check_linear_regression(...)`
- `check_logistic_regression(...)`

The examples are organized around three common workflows:

1. A clean example
2. A deliberately problematic example
3. A full-diagnostics example with `show_all=True`

Run the cells to see the terminal-style output and any relevant plots.

> Note
>
> The current MVP uses conservative outlier and influence heuristics. In practice, that means the clean linear and logistic examples below may still produce a `WARN` on influential points even when the rest of the diagnostics look good. That behavior is useful to see in context, so the notebook keeps those examples explicit rather than hiding them.

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd

from assumpcheck import (
    check_anova,
    check_linear_regression,
    check_logistic_regression,
)


def status_table(report_dict):
    frame = pd.DataFrame(report_dict["checks"])
    columns = ["assumption", "status", "metric", "threshold"]
    return frame[columns]


## ANOVA

In [ ]:
# Clean ANOVA example: all four checks pass.
groups_anova_clean = np.repeat(["A", "B", "C"], 12)
base = np.array([9.80, 9.90, 10.00, 10.10, 10.20, 9.95, 10.05, 10.00, 9.85, 10.15, 9.92, 10.08])
y_anova_clean = np.concatenate([base, base + 0.40, base + 0.80]) + np.tile(np.linspace(-0.05, 0.05, 12), 3)

anova_clean = check_anova(
    y=y_anova_clean,
    groups=groups_anova_clean,
    design_independent=True,
    plots_on_fail=False,
    return_dict=True,
)
status_table(anova_clean)


In [ ]:
# Problematic ANOVA example: inject an extreme outlier and extra spread.
y_anova_fail = y_anova_clean.copy()
y_anova_fail[4] = 14.50
y_anova_fail[24:36] = y_anova_fail[24:36] + np.linspace(-0.45, 0.45, 12)

anova_fail = check_anova(
    y=y_anova_fail,
    groups=groups_anova_clean,
    design_independent=True,
    return_dict=True,
)
status_table(anova_fail)


In [ ]:
# Full ANOVA diagnostics: even when everything passes, you can inspect all details and plots.
check_anova(
    y=y_anova_clean,
    groups=groups_anova_clean,
    design_independent=True,
    show_all=True,
)


## Linear Regression

In [ ]:
# Clean linear regression example: this usually yields 5 PASS and 1 WARN under the current influence heuristic.
rng_linear_clean = np.random.default_rng(27)
X_linear_clean = pd.DataFrame({
    "x1": rng_linear_clean.normal(size=35),
    "x2": rng_linear_clean.normal(size=35),
})
y_linear_clean = 1.0 + 1.8 * X_linear_clean["x1"] - 0.6 * X_linear_clean["x2"] + rng_linear_clean.normal(scale=0.35, size=35)

linear_clean = check_linear_regression(
    X=X_linear_clean,
    y=y_linear_clean,
    design_independent=True,
    plots_on_fail=False,
    return_dict=True,
)
status_table(linear_clean)


In [ ]:
# Problematic linear regression example: nonlinear signal plus non-constant variance.
rng_linear_fail = np.random.default_rng(7)
x = np.linspace(-2.5, 2.5, 90)
X_linear_fail = pd.DataFrame({"x1": x})
noise = rng_linear_fail.normal(scale=0.20 + 0.20 * np.abs(x), size=x.size)
y_linear_fail = 1.0 + 1.8 * x + 0.9 * x**2 + noise

linear_fail = check_linear_regression(
    X=X_linear_fail,
    y=y_linear_fail,
    design_independent=True,
    return_dict=True,
)
status_table(linear_fail)


In [ ]:
# Full linear regression diagnostics.
check_linear_regression(
    X=X_linear_clean,
    y=y_linear_clean,
    design_independent=True,
    show_all=True,
)


## Logistic Regression

In [ ]:
# Clean logistic regression example: this usually yields 4 PASS and 1 WARN under the current influence heuristic.
rng_logistic_clean = np.random.default_rng(44)
X_logistic_clean = pd.DataFrame({
    "x1": rng_logistic_clean.normal(size=45),
    "x2": rng_logistic_clean.normal(size=45),
})
logits_clean = -0.2 + 0.9 * X_logistic_clean["x1"] - 0.5 * X_logistic_clean["x2"]
p_clean = 1 / (1 + np.exp(-logits_clean))
y_logistic_clean = rng_logistic_clean.binomial(1, p_clean)

logistic_clean = check_logistic_regression(
    X=X_logistic_clean,
    y=y_logistic_clean,
    design_independent=True,
    plots_on_fail=False,
    return_dict=True,
)
status_table(logistic_clean)


In [ ]:
# Problematic logistic regression example: near-perfect separation drives a clear failure.
x_sep = np.linspace(-2.5, 2.5, 80)
X_logistic_fail = pd.DataFrame({
    "x1": x_sep,
    "x2": np.sin(x_sep),
})
y_logistic_fail = (x_sep > 0.0).astype(int)

logistic_fail = check_logistic_regression(
    X=X_logistic_fail,
    y=y_logistic_fail,
    design_independent=True,
    return_dict=True,
)
status_table(logistic_fail)


In [ ]:
# Full logistic regression diagnostics.
check_logistic_regression(
    X=X_logistic_clean,
    y=y_logistic_clean,
    design_independent=True,
    show_all=True,
)
